In [3]:
import json
import joblib
from pathlib import Path

## Load the model
MODEL_DIR = Path(r"D:\STUDY\Data_Science_Courses\PROJECTS\14.NorthBridge TradeFlow\NorthBridge-TradeFlow\Phase_5_Data Science and Predictive Intelligence\notebook\models")

# Load saved model
pipeline = joblib.load(
    MODEL_DIR / "rejection_risk_pipeline.joblib"
)

# Load metadata
with open(MODEL_DIR / "model_metadata.json", "r") as f:
    metadata = json.load(f)

print("Model loaded successfully!")
print("\nPipeline:")
print(pipeline)

print("\nModel metadata:")
print(json.dumps(metadata, indent=2))

Model loaded successfully!

Pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['notional_amount',
                                                   'tenor_years']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['currency', 'product_type',
                                                   'counterparty_id'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

Model metadata:
{
  "model_name": "Logistic Regression",
  "model_version": "1.0.0",
  "threshold": 0.1,
  "features": [
    "notional_amount",
    "tenor_years",
    "currency",
    "product_type",
    "counterparty_id"
  ],
  "target": "simulated_rejected",
  "random_seed": 42,
  "sklearn_version": "1.9.1",
 

In [5]:
## Verify model inference
import pandas as pd
# Example input — synthetic test record
sample_trade = pd.DataFrame([{
    "notional_amount": 10000000,
    "tenor_years": 10,
    "currency": "GBP",
    "product_type": "IRS",
    "counterparty_id": "CP-001"
}])

# Predict probability
risk_score = pipeline.predict_proba(sample_trade)[0, 1]

# Apply saved threshold
threshold = metadata["threshold"]
risk_prediction = int(risk_score >= threshold)

print(f"Model Version: {metadata['model_version']}")
print(f"Risk Probability: {risk_score:.4f}")
print(f"Risk Prediction: {risk_prediction}")
print(f"Threshold: {threshold}")

Model Version: 1.0.0
Risk Probability: 0.0746
Risk Prediction: 0
Threshold: 0.1


## MLflow Experiment Tracking

In [7]:
### %pip install mlflow
import mlflow

print("MLflow Version:", mlflow.__version__)

MLflow Version: 3.18.0


c:\Users\singh\anaconda3\envs\NorthBridge_TradeFlow\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Set up MLflow Experiment

In [9]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Store MLflow experiments locally
tracking_dir = Path("mlruns").resolve()
tracking_dir.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(tracking_dir.as_uri())

# Create or select experiment
mlflow.set_experiment(
    "NorthBridge_Rejection_Risk"
)

print("MLflow tracking configured!")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name(
    "NorthBridge_Rejection_Risk"
).name)

2026/11/09 21:17:29 INFO mlflow.tracking.fluent: Experiment with name 'NorthBridge_Rejection_Risk' does not exist. Creating a new experiment.


MLflow tracking configured!
Tracking URI: file:///D:/STUDY/Data_Science_Courses/PROJECTS/14.NorthBridge%20TradeFlow/NorthBridge-TradeFlow/Phase_6_ML%20Engineering/mlruns
Experiment: NorthBridge_Rejection_Risk


In [10]:
## Log the trained model into MLflow
import mlflow.sklearn

with mlflow.start_run(run_name="Logistic_Regression_v1.0.0") as run:

    # Model parameters
    mlflow.log_param("model_name", metadata["model_name"])
    mlflow.log_param("model_version", metadata["model_version"])
    mlflow.log_param("threshold", metadata["threshold"])
    mlflow.log_param("data_type", metadata["data_type"])

    # Evaluation metrics
    for key, value in metadata.items():
        if key.startswith("test_") and isinstance(value, (int, float)):
            mlflow.log_metric(key, value)

    # Save metadata as an artifact
    mlflow.log_dict(metadata, "model_metadata.json")

    # Log the trained pipeline
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="rejection_risk_model"
    )

    print("Model logged successfully!")
    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

Model logged successfully!
Run ID: d9664af6a81444cc85e54362fb165b4b
Model URI: models:/m-9a6b3431da374f3f9e7545fb88c92c1b


In [11]:
## Verify the MLflow model
import numpy as np

# Load model from MLflow
mlflow_model = mlflow.sklearn.load_model(
    model_info.model_uri
)

# Compare predictions
original_score = pipeline.predict_proba(sample_trade)[0, 1]
mlflow_score = mlflow_model.predict_proba(sample_trade)[0, 1]

print(f"Original Model Score: {original_score:.6f}")
print(f"MLflow Model Score:   {mlflow_score:.6f}")

print(
    "Predictions Match:",
    np.isclose(original_score, mlflow_score, atol=1e-8)
)

Original Model Score: 0.074595
MLflow Model Score:   0.074595
Predictions Match: True
